# Perplexity API

**Perplexity** is an **answer engine**: instead of returning a list of links, it runs a live web search, reads the results, and returns a *written answer grounded in those sources — with citations*. Its **Sonar API** exposes that whole pipeline behind one **OpenAI-compatible** `chat/completions` endpoint, so a single call does *search + read + synthesize + cite* for you.

**Domain:** Proprietary Models & Coding AI  ·  **recommended addition**  ·  **runnable:** yes  ·  _live cells gate on `os.getenv("PPLX_API_KEY")`_

## 1. What & Why

Perplexity is best understood as **retrieval-augmented generation (RAG) as a managed service**. A normal LLM API answers from frozen training weights; Perplexity's **Sonar** models answer from **the live web**, every request:

1. Take the user question.
2. Issue real web searches behind the scenes.
3. Pull the top results into context.
4. Have an LLM (a fine-tuned Llama family model) write a grounded answer.
5. Return the answer **plus the source URLs it used** (`search_results` / `citations`).

**The problem it solves.** Plain LLMs hallucinate, go stale after their training cutoff, and can't cite. The usual fix is to build your *own* RAG stack — crawler, embeddings, vector DB, reranker, retrieval glue. Perplexity collapses all of that into one HTTP call: **you never manage an index**. You pay for fresh, cited, factual answers on tap.

**When to reach for it**

- Questions whose answers **change over time** or post-date a model's training cutoff (news, prices, releases, scores, docs for a new library version).
- You want answers that **cite their sources** so a human (or downstream system) can verify.
- You want web-grounded Q&A **without building or operating a retrieval pipeline**.

**When *not* to.** Perplexity searches the web on **every** request and bills a per-request search fee on top of tokens. So it's the wrong tool for: offline/private-data reasoning (no web grounding helps), pure creative writing, bulk transformation of text you already have, or **agentic code generation** — despite living in the "coding AI" domain, Sonar is an *answer engine*, not a coding copilot. For those, a plain LLM API (Claude, GPT, a self-hosted Llama) is cheaper and better.

## 2. Mental Model

Think of Perplexity as **"a search engine and an LLM fused into one endpoint — you ask, it googles, reads, and writes you a cited paragraph."**

```
         your question
              │
              ▼
   ┌─────────────────────────────────────────────┐
   │            Perplexity Sonar API              │
   │                                              │
   │   ┌─────────┐   ┌──────────┐   ┌──────────┐  │
   │   │  web    │──▶│ fetch &  │──▶│  Sonar   │  │
   │   │ search  │   │ read top │   │  LLM      │  │
   │   │ (live)  │   │ results  │   │ (Llama)  │  │
   │   └─────────┘   └──────────┘   └────┬─────┘  │
   │     ▲ you pay a per-request           │       │
   │     │ SEARCH fee here                 │       │
   └─────┼─────────────────────────────────┼───────┘
         │                                 ▼
   search_domain_filter,           answer text  +  citations[]
   search_recency_filter,                          search_results[]
   search_mode                     (you pay TOKEN fees here)
```

Three things to internalize:

1. **One call = RAG.** The search, retrieval, and generation all happen server-side. You send messages; you get back prose **+ a list of sources**. You never touch a vector DB.
2. **OpenAI-shaped, so it drops into existing code.** Same `chat/completions` body, same `OpenAI` SDK with a different `base_url`. The *extra* you get back is `citations` / `search_results`; the *extra* you can send are the `search_*` knobs.
3. **You pay twice: tokens *and* search.** Every request incurs a search fee (sized by how much web context you ask for) on top of input/output token costs. Budgeting like a plain LLM under-counts.

## 3. Key Concepts

- **Answer engine vs search engine.** A search engine returns *links*; an answer engine returns a *synthesized, cited answer*. Perplexity is the latter, exposed via API.
- **Sonar models.** The API model family. The current lineup:
  - `sonar` — fast, cheap, web-grounded Q&A (the default workhorse).
  - `sonar-pro` — larger context + more thorough search for complex/multi-step questions.
  - `sonar-reasoning` / `sonar-reasoning-pro` — chain-of-thought reasoning *over* live search (emit a `<think>` block before the answer).
  - `sonar-deep-research` — runs many searches and writes a long, exhaustive report (slow, expensive; minutes per call).
- **Citations / `search_results`.** Every response includes the sources used: `search_results` is the structured list (`title`, `url`, `date`), and `citations` is the flat URL list. **Surfacing these to users is the whole point** (and expected by Perplexity's terms).
- **Search controls.** Shape what gets retrieved:
  - `search_domain_filter` — allow/deny-list domains (e.g. `["arxiv.org", "-reddit.com"]`).
  - `search_recency_filter` — `"day" | "week" | "month" | "year"` to bias toward fresh results.
  - `search_mode` — `"web"` (default) or `"academic"` (scholarly sources).
  - `web_search_options.search_context_size` — `"low" | "medium" | "high"`: how much web context to pull. **Higher = better grounding but a bigger search fee.**
- **OpenAI compatibility.** Endpoint is `https://api.perplexity.ai/chat/completions`; auth is `Authorization: Bearer $PPLX_API_KEY`. Works with the official `openai` SDK by setting `base_url`.
- **Structured outputs.** Sonar supports `response_format` (JSON schema / regex) so you can get machine-parseable, *cited* data — grounded extraction in one call.
- **Two-part pricing.** Per-million **input/output tokens** (varies by model) **plus** a **per-request search fee** that scales with `search_context_size`. Reasoning models also bill their `<think>` tokens.

## 4. Setup

You need a Perplexity account, then an **API key** from the dashboard (it's a paid product — API access is billed pay-as-you-go, separate from the consumer Pro subscription).

```bash
# 1. Get a key:  https://www.perplexity.ai/settings/api  (add credit / a card)
# 2. Export it (the notebook reads PPLX_API_KEY):
export PPLX_API_KEY="pplx-..."

# 3. Optional SDK — Sonar is OpenAI-compatible, so the OpenAI client just works:
pip install openai          # or call the REST endpoint with plain urllib (shown below)
```

Minimal call shape with the OpenAI SDK (gated below so the notebook always runs):

```python
from openai import OpenAI
client = OpenAI(api_key=os.environ["PPLX_API_KEY"],
                base_url="https://api.perplexity.ai")
resp = client.chat.completions.create(
    model="sonar",
    messages=[{"role": "user", "content": "What shipped in Python 3.13?"}],
)
print(resp.choices[0].message.content)
print(resp.citations)          # the sources it used
```

The cells below run top-to-bottom in a fresh kernel **without** the `openai` SDK or an API key: every network call is gated behind an `os.getenv` check, while the parsing/estimation examples always execute.

In [ ]:
# This notebook runs with or without an API key or the openai SDK.
# To run the live example:  export PPLX_API_KEY="pplx-..."   (optionally pip install openai)
import os

api_key = os.getenv("PPLX_API_KEY")

try:
    import openai  # noqa: F401
    have_sdk = True
except ImportError:
    have_sdk = False

print("PPLX_API_KEY :", "set" if api_key else "(unset — live API calls skipped)")
print("openai SDK   :", "installed" if have_sdk else "(not installed — urllib fallback used)")
print("\nSonar models: sonar, sonar-pro, sonar-reasoning(-pro), sonar-deep-research")
print("Endpoint     : https://api.perplexity.ai/chat/completions  (OpenAI-compatible)")

## 5. Worked Examples

### Example 1 — Anatomy of a Sonar response (no network)

The thing that makes Perplexity *Perplexity* isn't the answer text — it's the **sources** that come back with it. Below is a representative Sonar JSON payload (trimmed). Parsing it by hand puts the response shape in muscle memory: the prose lives where any OpenAI client expects it, and the grounding lives in `search_results` / `citations`.

In [ ]:
# A representative Sonar chat/completions response (trimmed, OpenAI-shaped).
sample_response = {
    "id": "cmpl-abc123",
    "model": "sonar",
    "choices": [{
        "index": 0,
        "finish_reason": "stop",
        "message": {
            "role": "assistant",
            "content": (
                "Python 3.13 shipped an experimental free-threaded (no-GIL) build "
                "and a new interactive REPL [1][2]."
            ),
        },
    }],
    "citations": [
        "https://docs.python.org/3.13/whatsnew/3.13.html",
        "https://peps.python.org/pep-0703/",
    ],
    "search_results": [
        {"title": "What's New in Python 3.13",
         "url": "https://docs.python.org/3.13/whatsnew/3.13.html", "date": "2024-10-07"},
        {"title": "PEP 703 – Making the GIL Optional",
         "url": "https://peps.python.org/pep-0703/", "date": "2024-10-01"},
    ],
    "usage": {"prompt_tokens": 18, "completion_tokens": 31, "total_tokens": 49},
}

# 1) The answer is exactly where any OpenAI client looks:
answer = sample_response["choices"][0]["message"]["content"]
print("ANSWER:\n ", answer, "\n")

# 2) The grounding is the payload's whole reason to exist — render it:
print("SOURCES:")
for i, src in enumerate(sample_response["search_results"], start=1):
    print(f"  [{i}] {src['title']}  ({src['date']})")
    print(f"      {src['url']}")

# 3) usage drives the TOKEN half of the bill (search fee is billed separately):
u = sample_response["usage"]
print(f"\ntokens: {u['prompt_tokens']} in + {u['completion_tokens']} out = {u['total_tokens']}")

### Example 2 — Estimate the *real* cost: tokens **plus** the search fee (no network)

The #1 budgeting mistake is pricing Sonar like a plain LLM and forgetting the **per-request search fee**. Cost = token cost **+** a flat search fee that scales with `search_context_size`. The estimator below makes the two halves explicit. (Rates are illustrative — always check the live pricing page; the *structure* is the lesson.)

In [ ]:
# Sonar cost = token cost + per-request SEARCH fee. Illustrative rates ($/1M tokens).
TOKEN_RATES = {                       # (input $/M, output $/M)
    "sonar":            (1.0,  1.0),
    "sonar-pro":        (3.0, 15.0),
    "sonar-reasoning":  (1.0,  5.0),
}
# Per-request search fee in dollars, by search_context_size (illustrative).
SEARCH_FEE = {"low": 0.005, "medium": 0.008, "high": 0.012}

def request_cost(model, in_tok, out_tok, context_size="medium"):
    in_rate, out_rate = TOKEN_RATES[model]
    token_cost  = in_tok / 1_000_000 * in_rate + out_tok / 1_000_000 * out_rate
    search_cost = SEARCH_FEE[context_size]
    return token_cost, search_cost

# A typical small Q&A: 200 input tokens, 400 output tokens, medium grounding.
for model in ("sonar", "sonar-pro"):
    tok, search = request_cost(model, in_tok=200, out_tok=400, context_size="medium")
    total = tok + search
    pct = search / total * 100
    print(f"{model:12s}: tokens ${tok:.5f} + search ${search:.5f} "
          f"= ${total:.5f}/req  (search = {pct:.0f}% of cost)")

print("\nLesson: at small token counts the SEARCH fee dominates the bill —")
print("so 1,000 short answers cost ~the same as far fewer long ones. Budget per-REQUEST,")
print("not just per-token, and drop search_context_size to 'low' when grounding is easy.")

### Example 3 — Call the live Sonar API (gated on `PPLX_API_KEY`)

The real thing. With `PPLX_API_KEY` set this sends a live, web-grounded request and prints the answer **and its citations**; otherwise it prints the exact call shape so the notebook still executes cleanly. We use plain `urllib` (no SDK required) — the body is OpenAI-compatible, so swapping in the `openai` client is a one-liner.

In [ ]:
# Live, web-grounded call — gated so the notebook runs with or without a key.
import os, json, urllib.request, urllib.error

ENDPOINT = "https://api.perplexity.ai/chat/completions"

def sonar_ask(question, model="sonar", recency="month"):
    body = json.dumps({
        "model": model,
        "messages": [
            {"role": "system", "content": "Answer in one concise sentence."},
            {"role": "user",   "content": question},
        ],
        "search_recency_filter": recency,            # bias toward fresh sources
        "web_search_options": {"search_context_size": "low"},   # cheaper grounding
    }).encode()
    req = urllib.request.Request(
        ENDPOINT, data=body,
        headers={"Authorization": f"Bearer {os.environ['PPLX_API_KEY']}",
                 "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

if api_key:
    try:
        data = sonar_ask("What is the latest stable version of Python?")
        print("ANSWER:", data["choices"][0]["message"]["content"].strip())
        print("\nSOURCES:")
        for i, url in enumerate(data.get("citations", []), start=1):
            print(f"  [{i}] {url}")
    except urllib.error.HTTPError as e:              # auth / quota / bad request
        print("Live call failed:", e.code, e.read().decode()[:200])
    except Exception as e:
        print("Live call failed:", type(e).__name__, e)
else:
    print("Skipping live API call (set PPLX_API_KEY to run).")
    print("Request body would be:")
    print("  POST https://api.perplexity.ai/chat/completions")
    print('  {"model": "sonar", "messages": [...], "search_recency_filter": "month"}')
    print("  -> response.choices[0].message.content   + response['citations']")

## 6. Gotchas & Pitfalls

- **You pay a search fee on *every* request.** Cost ≠ tokens alone. For short answers the per-request search fee often **dominates** the bill (see Example 2). Budget per-request, and lower `search_context_size` to `"low"` when the question is easy.
- **It's not a coding model.** Despite the "coding AI" domain, Sonar is an *answer engine*. Don't reach for it to write/refactor code or drive an agent loop — use Claude/GPT/a self-hosted model. Sonar shines for *looking things up*, not *building*.
- **Don't use it for offline or private data.** Every request hits the web. For questions over your own documents, build real RAG or use a plain LLM with your context — Sonar can't search what isn't public.
- **Surface the citations.** The sources aren't decoration; Perplexity's terms expect you to show them, and they're how anyone verifies the answer. Dropping `citations` defeats the purpose.
- **Reasoning models emit a `<think>` block.** `sonar-reasoning(-pro)` prepend chain-of-thought before the answer (and bill those tokens). Strip the `<think>...</think>` prefix before displaying, and budget for the extra tokens.
- **Stale/deprecated model names.** The old `llama-3.1-sonar-small-128k-online`-style names are retired — use `sonar`, `sonar-pro`, etc. Hard-coded legacy names return errors.
- **System prompt ≠ search query.** The `system` message shapes *tone/format*, not *what gets searched*. Retrieval is driven by the **user** message — put the actual question there.
- **Freshness still isn't real-time-guaranteed.** `search_recency_filter` biases toward recent results but indexing lag means the very latest events may not appear; don't treat it as a newswire.
- **No fine-tuning, limited determinism.** You can't fine-tune Sonar, and because the live web changes under you, the *same* question can return different answers day to day. Don't build exact-match assertions on its output.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Web-grounded, *cited* answers with zero RAG plumbing** | **Perplexity Sonar** | One call does search + read + synthesize + cite. No index to build or operate. |
| **Answers to questions newer than a model's training cutoff** | **Perplexity Sonar** | Live web every request; `search_recency_filter` biases toward fresh sources. |
| **A long, exhaustive researched report** | **`sonar-deep-research`** | Runs many searches and writes a structured report (slow/pricey, but hands-off). |
| **Grounding over *your own* private docs** | **Build RAG** (LangChain/LlamaIndex + a vector DB) or an LLM with your context | Sonar only searches the public web; it can't see your data. |
| **Code generation / agents / tool use** | **Claude / GPT / self-hosted Llama** | Sonar is an answer engine, not a coding or agent model. |
| **Cheap, high-volume text transforms you already have the text for** | **A plain LLM API** | No search needed — avoid paying Perplexity's per-request search fee. |
| **Web search as a *tool* inside your own agent** | **Native web-search tools** (Claude/OpenAI built-in search, or a search API like Tavily/Brave/Exa) | Keep your model + orchestration; bolt on retrieval where *you* control it. |

**Honest trade-offs**

- **vs building your own RAG** — Perplexity wins on *time-to-value* (nothing to operate) and freshness; you give up control over the corpus, ranking, and chunking, and you can't ground on private data. Heavy/specialized retrieval still wants a custom stack.
- **vs a frontier LLM's built-in web search** (Claude/GPT tool use) — those keep you on one model with full orchestration control and let you mix search with other tools/code; Perplexity is the more *turnkey* "just give me a cited answer" path, with answer-engine-tuned models.
- **vs a bare search API** (Tavily, Brave, Exa, Bing) — those return raw results for *you* to read and synthesize; Perplexity also does the synthesis + citation. Pick a search API when you want to feed results into your own model/prompt; pick Sonar when you want the finished answer.

## 8. Resources

- **Perplexity API docs (Sonar)** — https://docs.perplexity.ai/
- **API quickstart / first call** — https://docs.perplexity.ai/guides/getting-started
- **Models & capabilities** — https://docs.perplexity.ai/guides/model-cards
- **Search controls (domain/recency/academic filters)** — https://docs.perplexity.ai/guides/search-domain-filters
- **Pricing (tokens + per-request search fee)** — https://docs.perplexity.ai/guides/pricing
- **Structured outputs (`response_format`)** — https://docs.perplexity.ai/guides/structured-outputs
- **API key dashboard** — https://www.perplexity.ai/settings/api

**Related notebooks:** `anthropic-claude-api`, `google-gemini`, `grok` (frontier LLMs with their *own* web-search tools — the build-your-own-grounding alternative); `mistral`, `llama` (self-hosted models you'd pair with a custom RAG stack instead).

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def sonar_bill(requests, in_rate, out_rate, search_fees):
    """Split a grounded-answer bill into its token half and its per-request half."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE